# Proyek Klasifikasi Gambar: Fruit Classification

**Nama:** [Input Nama Anda]

**Email:** [Input Email Anda]

**ID Dicoding:** [Input Username Dicoding Anda]

---

## Deskripsi Proyek
Proyek ini mengklasifikasikan gambar buah-buahan menggunakan Convolutional Neural Network (CNN) dengan arsitektur Sequential.

## 1. Import Library

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os
import shutil
from sklearn.model_selection import train_test_split
from PIL import Image
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow Version: {tf.__version__}")

## 2. Download Dataset dari Kaggle

In [ ]:
# Install kaggle
!pip install -q kaggle

In [ ]:
# Upload kaggle.json
# Dapatkan dari: Kaggle > Profile > Settings > API > Create New Token
from google.colab import files
print("Upload file kaggle.json dari komputer Anda:")
files.upload()

In [ ]:
# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle credentials berhasil di-setup!")

In [ ]:
# Download Fruits 360 Dataset
!kaggle datasets download -d moltean/fruits
print("\n✅ Download selesai!")

In [ ]:
# Extract dataset
import zipfile

with zipfile.ZipFile('fruits.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/fruits_dataset')

print("✅ Dataset berhasil diekstrak!")

# Lihat struktur
!ls /content/fruits_dataset/

## 3. Konfigurasi Path Dataset

In [ ]:
# PATH YANG BENAR untuk Fruits 360 Dataset
SOURCE_DIR = '/content/fruits_dataset/fruits-360_original-size/fruits-360-original-size/Training'

# Path alternatif jika yang atas tidak ada
if not os.path.exists(SOURCE_DIR):
    SOURCE_DIR = '/content/fruits_dataset/fruits-360_100x100/fruits-360/Training'

if not os.path.exists(SOURCE_DIR):
    # Cari secara otomatis
    for root, dirs, files in os.walk('/content/fruits_dataset'):
        if 'Training' in dirs:
            potential_path = os.path.join(root, 'Training')
            if len(os.listdir(potential_path)) > 10:
                SOURCE_DIR = potential_path
                break

print(f"✅ SOURCE_DIR: {SOURCE_DIR}")
print(f"   Exists: {os.path.exists(SOURCE_DIR)}")

In [ ]:
# Lihat kelas yang tersedia
print("=" * 60)
print("KELAS YANG TERSEDIA")
print("=" * 60)

available_classes = sorted(os.listdir(SOURCE_DIR))
print(f"\nTotal kelas: {len(available_classes)}\n")

for i, cls in enumerate(available_classes):
    class_path = os.path.join(SOURCE_DIR, cls)
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        print(f"  {i+1:3}. {cls} ({count} gambar)")

## 4. Split Dataset (Train, Validation, Test)

In [ ]:
# ============================================
# PILIH KELAS YANG INGIN DIGUNAKAN
# Sesuaikan dengan nama kelas yang tersedia!
# ============================================

CLASSES = ['Apple Red 1', 'Banana', 'Orange']

# Path output
DATASET_DIR = '/content/dataset'
TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VAL_DIR = os.path.join(DATASET_DIR, 'validation')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

print(f"Kelas yang dipilih: {CLASSES}")

In [ ]:
# Hapus folder lama jika ada
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
    print("🗑️ Folder lama dihapus")

# Buat struktur folder baru
for split in ['train', 'validation', 'test']:
    for cls in CLASSES:
        path = os.path.join(DATASET_DIR, split, cls)
        os.makedirs(path, exist_ok=True)

print("✅ Struktur folder baru dibuat")

In [ ]:
# Split dan copy data
print("=" * 60)
print("SPLITTING DATASET (70% Train, 15% Val, 15% Test)")
print("=" * 60)

total_images = 0

for cls in CLASSES:
    source_class_dir = os.path.join(SOURCE_DIR, cls)
    
    if not os.path.exists(source_class_dir):
        print(f"\n❌ Kelas '{cls}' tidak ditemukan!")
        continue
    
    # Ambil semua file gambar
    images = [f for f in os.listdir(source_class_dir) 
              if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
    
    print(f"\n📂 {cls}: {len(images)} gambar")
    total_images += len(images)
    
    # Split data
    train_imgs, temp_imgs = train_test_split(images, train_size=0.7, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)
    
    # Copy ke folder train
    for img in train_imgs:
        shutil.copy2(os.path.join(source_class_dir, img), os.path.join(TRAIN_DIR, cls, img))
    
    # Copy ke folder validation
    for img in val_imgs:
        shutil.copy2(os.path.join(source_class_dir, img), os.path.join(VAL_DIR, cls, img))
    
    # Copy ke folder test
    for img in test_imgs:
        shutil.copy2(os.path.join(source_class_dir, img), os.path.join(TEST_DIR, cls, img))
    
    print(f"   ├── Train: {len(train_imgs)}")
    print(f"   ├── Val: {len(val_imgs)}")
    print(f"   └── Test: {len(test_imgs)}")

print("\n" + "=" * 60)
print(f"✅ TOTAL: {total_images} gambar berhasil di-split!")
print("=" * 60)

In [ ]:
# Verifikasi hasil split
def count_images(directory):
    total = 0
    if os.path.exists(directory):
        for cls in os.listdir(directory):
            class_dir = os.path.join(directory, cls)
            if os.path.isdir(class_dir):
                count = len(os.listdir(class_dir))
                print(f"  {cls}: {count} gambar")
                total += count
    return total

print("=" * 50)
print("VERIFIKASI DATASET")
print("=" * 50)

print("\n📁 Training Set:")
train_count = count_images(TRAIN_DIR)
print(f"Total: {train_count}\n")

print("📁 Validation Set:")
val_count = count_images(VAL_DIR)
print(f"Total: {val_count}\n")

print("📁 Test Set:")
test_count = count_images(TEST_DIR)
print(f"Total: {test_count}\n")

total_all = train_count + val_count + test_count
print("=" * 50)
print(f"🎯 GRAND TOTAL: {total_all} gambar")

if total_all >= 1000:
    print("✅ Kriteria terpenuhi: >= 1000 gambar")
else:
    print(f"⚠️ Perlu tambah {1000 - total_all} gambar lagi")
print("=" * 50)

## 5. Cek Resolusi Gambar

In [ ]:
# Fungsi untuk cek resolusi (dari tips Dicoding)
def print_images_resolution(directory):
    unique_sizes = set()
    total_images = 0

    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)
        if os.path.isdir(subdir_path):
            image_files = os.listdir(subdir_path)
            num_images = len(image_files)
            print(f"{subdir}: {num_images}")
            total_images += num_images

            for img_file in image_files[:10]:
                img_path = os.path.join(subdir_path, img_file)
                try:
                    with Image.open(img_path) as img:
                        unique_sizes.add(img.size)
                except:
                    pass

            for size in unique_sizes:
                print(f"- {size}")
            print("---------------")
            unique_sizes.clear()

    print(f"\nTotal: {total_images}")

print("Resolusi gambar di Training Set:")
print_images_resolution(TRAIN_DIR)

## 6. Data Preprocessing dan Augmentation

In [ ]:
# Parameter
IMG_HEIGHT = 150
IMG_WIDTH = 150
BATCH_SIZE = 32

# ImageDataGenerator untuk Training (dengan augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# ImageDataGenerator untuk Validation dan Test (tanpa augmentation)
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("✅ ImageDataGenerator berhasil dibuat!")

In [ ]:
# Generator untuk Training
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

# Generator untuk Validation
validation_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Generator untuk Test
test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Info
print(f"\n✅ Train samples: {train_generator.samples}")
print(f"✅ Validation samples: {validation_generator.samples}")
print(f"✅ Test samples: {test_generator.samples}")

# Simpan labels
labels = list(train_generator.class_indices.keys())
class_indices = train_generator.class_indices
NUM_CLASSES = len(class_indices)

print(f"\n✅ Labels: {labels}")
print(f"✅ Num classes: {NUM_CLASSES}")

## 7. Visualisasi Sample Data

In [ ]:
def visualize_samples(generator, title="Sample Images"):
    plt.figure(figsize=(15, 10))
    images, labels_batch = next(generator)
    class_names = list(generator.class_indices.keys())
    
    for i in range(min(12, len(images))):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i])
        label_idx = np.argmax(labels_batch[i])
        plt.title(class_names[label_idx])
        plt.axis('off')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_samples(train_generator, "Sample Training Images (dengan Augmentation)")

## 8. Membangun Model CNN

In [ ]:
print(f"Jumlah Kelas: {NUM_CLASSES}")

# Model Sequential dengan Conv2D dan MaxPooling
model = Sequential([
    # Block 1
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    
    # Block 2
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    # Block 3
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    # Block 4
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    # Flatten dan Dense layers
    Flatten(),
    Dropout(0.5),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 9. Implementasi Callbacks

In [ ]:
# Callback 1: Early Stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Callback 2: Model Checkpoint
checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Callback 3: Reduce LR on Plateau
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

callbacks = [early_stopping, checkpoint, reduce_lr]
print("✅ Callbacks berhasil dibuat!")

## 10. Training Model

In [ ]:
EPOCHS = 50

history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training selesai!")

## 11. Plot Akurasi dan Loss

In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot Accuracy
    axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[0].set_title('Model Accuracy', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend(loc='lower right')
    axes[0].grid(True, alpha=0.3)
    
    # Plot Loss
    axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[1].set_title('Model Loss', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ Plot disimpan sebagai 'training_history.png'")

plot_training_history(history)

## 12. Evaluasi Model

In [ ]:
print("=" * 50)
print("EVALUASI MODEL")
print("=" * 50)

# Evaluasi Training Set
train_generator.reset()
train_loss, train_accuracy = model.evaluate(train_generator, verbose=0)
print(f"\nTraining Set:")
print(f"  Loss: {train_loss:.4f}")
print(f"  Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")

# Evaluasi Validation Set
validation_generator.reset()
val_loss, val_accuracy = model.evaluate(validation_generator, verbose=0)
print(f"\nValidation Set:")
print(f"  Loss: {val_loss:.4f}")
print(f"  Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")

# Evaluasi Test Set
test_generator.reset()
test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
print(f"\nTest Set:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

print("\n" + "=" * 50)
if train_accuracy >= 0.85 and test_accuracy >= 0.85:
    print("✅ KRITERIA TERPENUHI: Akurasi >= 85%")
else:
    print("❌ KRITERIA BELUM TERPENUHI: Akurasi < 85%")
print("=" * 50)

## 13. Confusion Matrix dan Classification Report

In [ ]:
# Prediksi
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification Report
print("\nClassification Report:")
print("=" * 60)
print(classification_report(true_classes, predicted_classes, target_names=class_labels))

## 14. Simpan Model - SavedModel

In [ ]:
SAVED_MODEL_DIR = 'saved_model'

model.save(SAVED_MODEL_DIR)

print(f"✅ Model berhasil disimpan dalam format SavedModel di: {SAVED_MODEL_DIR}")

# Lihat isi direktori
print("\nIsi direktori SavedModel:")
for root, dirs, files in os.walk(SAVED_MODEL_DIR):
    level = root.replace(SAVED_MODEL_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

## 15. Konversi ke TF-Lite

In [ ]:
TFLITE_DIR = 'tflite'
os.makedirs(TFLITE_DIR, exist_ok=True)

# Konversi
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Simpan model
tflite_model_path = os.path.join(TFLITE_DIR, 'model.tflite')
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

print(f"✅ Model TF-Lite disimpan di: {tflite_model_path}")
print(f"   Ukuran: {os.path.getsize(tflite_model_path) / (1024*1024):.2f} MB")

# Simpan label.txt
label_path = os.path.join(TFLITE_DIR, 'label.txt')
with open(label_path, 'w') as f:
    for label in labels:
        f.write(f"{label}\n")

print(f"✅ Label disimpan di: {label_path}")
print("\nIsi label.txt:")
!cat {label_path}

## 16. Konversi ke TFJS

In [ ]:
# Install tensorflowjs
!pip install tensorflowjs -q

TFJS_DIR = 'tfjs_model'

# Konversi
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model {SAVED_MODEL_DIR} {TFJS_DIR}

print(f"\n✅ Model TFJS disimpan di: {TFJS_DIR}")

# Lihat isi
print("\nIsi direktori TFJS:")
for file in os.listdir(TFJS_DIR):
    file_path = os.path.join(TFJS_DIR, file)
    size = os.path.getsize(file_path) / 1024
    print(f"  {file}: {size:.2f} KB")

## 17. Inference dengan SavedModel

In [ ]:
print("=" * 60)
print("INFERENCE MENGGUNAKAN SAVEDMODEL")
print("=" * 60)

# Load model
loaded_model = tf.keras.models.load_model(SAVED_MODEL_DIR)

# Fungsi preprocessing
def preprocess_image(image_path, target_size=(150, 150)):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=target_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

# Ambil sample dari test set
test_images = []
for cls in os.listdir(TEST_DIR):
    class_dir = os.path.join(TEST_DIR, cls)
    if os.path.isdir(class_dir):
        images = os.listdir(class_dir)
        if images:
            test_images.append((os.path.join(class_dir, images[0]), cls))

# Inference
print("\nHasil Inference SavedModel:")
print("-" * 60)

for img_path, true_label in test_images:
    if os.path.exists(img_path):
        img_array = preprocess_image(img_path)
        predictions = loaded_model.predict(img_array, verbose=0)
        predicted_class = labels[np.argmax(predictions[0])]
        confidence = np.max(predictions[0]) * 100
        
        status = "✅" if predicted_class == true_label else "❌"
        print(f"{status} Gambar: {os.path.basename(img_path)}")
        print(f"   True Label: {true_label}")
        print(f"   Prediksi: {predicted_class}")
        print(f"   Confidence: {confidence:.2f}%")
        print()

## 18. Inference dengan TF-Lite

In [ ]:
print("=" * 60)
print("INFERENCE MENGGUNAKAN TF-LITE")
print("=" * 60)

# Load TF-Lite model
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\nInput shape: {input_details[0]['shape']}")
print(f"Output shape: {output_details[0]['shape']}")

# Fungsi inference TF-Lite
def tflite_inference(interpreter, image_path, target_size=(150, 150)):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=target_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0).astype(np.float32)
    
    interpreter.set_tensor(input_details[0]['index'], img_array)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    return output

# Inference
print("\nHasil Inference TF-Lite:")
print("-" * 60)

for img_path, true_label in test_images:
    if os.path.exists(img_path):
        predictions = tflite_inference(interpreter, img_path)
        predicted_class = labels[np.argmax(predictions[0])]
        confidence = np.max(predictions[0]) * 100
        
        status = "✅" if predicted_class == true_label else "❌"
        print(f"{status} Gambar: {os.path.basename(img_path)}")
        print(f"   True Label: {true_label}")
        print(f"   Prediksi: {predicted_class}")
        print(f"   Confidence: {confidence:.2f}%")
        print()

## 19. Visualisasi Hasil Prediksi

In [ ]:
def visualize_predictions(model, test_dir, labels, num_images=9):
    plt.figure(figsize=(15, 15))
    
    all_images = []
    for cls in os.listdir(test_dir):
        class_dir = os.path.join(test_dir, cls)
        if os.path.isdir(class_dir):
            for img_name in os.listdir(class_dir)[:3]:
                all_images.append((os.path.join(class_dir, img_name), cls))
    
    sample_images = all_images[:num_images]
    
    for i, (img_path, true_label) in enumerate(sample_images):
        plt.subplot(3, 3, i + 1)
        
        img = tf.keras.preprocessing.image.load_img(img_path, target_size=(150, 150))
        plt.imshow(img)
        
        img_array = preprocess_image(img_path)
        predictions = model.predict(img_array, verbose=0)
        predicted_class = labels[np.argmax(predictions[0])]
        confidence = np.max(predictions[0]) * 100
        
        color = 'green' if predicted_class == true_label else 'red'
        plt.title(f"True: {true_label}\nPred: {predicted_class} ({confidence:.1f}%)", 
                  color=color, fontsize=10)
        plt.axis('off')
    
    plt.suptitle('Hasil Prediksi Model', fontsize=16)
    plt.tight_layout()
    plt.savefig('prediction_results.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_predictions(loaded_model, TEST_DIR, labels)

## 20. Buat ZIP untuk Submission

In [ ]:
# Buat folder submission
submission_dir = 'submission'
os.makedirs(submission_dir, exist_ok=True)

# Copy model files
if os.path.exists(SAVED_MODEL_DIR):
    shutil.copytree(SAVED_MODEL_DIR, os.path.join(submission_dir, 'saved_model'), dirs_exist_ok=True)

if os.path.exists(TFLITE_DIR):
    shutil.copytree(TFLITE_DIR, os.path.join(submission_dir, 'tflite'), dirs_exist_ok=True)

if os.path.exists(TFJS_DIR):
    shutil.copytree(TFJS_DIR, os.path.join(submission_dir, 'tfjs_model'), dirs_exist_ok=True)

# Buat requirements.txt
with open(os.path.join(submission_dir, 'requirements.txt'), 'w') as f:
    f.write("""tensorflow>=2.10.0
numpy>=1.21.0
matplotlib>=3.5.0
Pillow>=9.0.0
scikit-learn>=1.0.0
seaborn>=0.11.0
tensorflowjs>=4.0.0""")

print("✅ Semua file berhasil dicopy ke folder submission")

# Lihat struktur
print("\nStruktur folder submission:")
for root, dirs, files in os.walk(submission_dir):
    level = root.replace(submission_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

In [ ]:
# Buat ZIP
shutil.make_archive('submission', 'zip', submission_dir)
print("✅ File submission.zip berhasil dibuat!")
print(f"   Ukuran: {os.path.getsize('submission.zip') / (1024*1024):.2f} MB")

In [ ]:
# Download ZIP
from google.colab import files
files.download('submission.zip')

## 21. Summary

In [ ]:
print("=" * 70)
print("SUMMARY PROYEK KLASIFIKASI GAMBAR")
print("=" * 70)

print("\n📊 DATASET:")
print(f"   - Total Gambar: {train_count + val_count + test_count}")
print(f"   - Training: {train_count}")
print(f"   - Validation: {val_count}")
print(f"   - Testing: {test_count}")
print(f"   - Jumlah Kelas: {NUM_CLASSES} ({', '.join(labels)})")

print("\n🏗️ ARSITEKTUR MODEL:")
print(f"   - Model: Sequential")
print(f"   - Layers: Conv2D + MaxPooling2D + Dense")
print(f"   - Total Parameters: {model.count_params():,}")

print("\n📈 PERFORMA MODEL:")
print(f"   - Training Accuracy: {train_accuracy*100:.2f}%")
print(f"   - Validation Accuracy: {val_accuracy*100:.2f}%")
print(f"   - Test Accuracy: {test_accuracy*100:.2f}%")

print("\n💾 MODEL TERSIMPAN:")
print(f"   - SavedModel: ✅")
print(f"   - TF-Lite: ✅")
print(f"   - TFJS: ✅")

print("\n✅ KRITERIA SUBMISSION:")
print(f"   [{'✓' if train_count + val_count + test_count >= 1000 else '✗'}] Minimal 1000 gambar")
print(f"   [{'✓' if NUM_CLASSES >= 3 else '✗'}] Minimal 3 kelas")
print(f"   [✓] Dataset dibagi: Train, Validation, Test")
print(f"   [✓] Model Sequential dengan Conv2D dan Pooling")
print(f"   [{'✓' if train_accuracy >= 0.85 and test_accuracy >= 0.85 else '✗'}] Akurasi >= 85%")
print(f"   [✓] Plot akurasi dan loss")
print(f"   [✓] Model: SavedModel, TF-Lite, TFJS")
print(f"   [✓] Implementasi Callback")
print(f"   [✓] Inference dengan bukti")

print("\n" + "=" * 70)
print("🎉 PROYEK SELESAI!")
print("=" * 70)